# Music generation with GPT-2

<a href="https://colab.research.google.com/github/abinaya25138/midi-music-generation-gpt2/blob/main/music_generation.ipynb"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open in Colab"></a>

This notebook trains a small GPT-2 from scratch to write multi-track MIDI music.

1. MIDI files are cut into 8-bar pieces.
2. Each piece is written out as text in the MMM token format (`NOTE_ON=60 TIME_DELTA=4 NOTE_OFF=60 ...`).
3. A word-level tokenizer and a GPT-2 language model are trained on that text.
4. After every evaluation the model composes a short piece, which is played back as audio so you can hear it improve.

Use a GPU runtime (**Runtime → Change runtime type → GPU**).

## 1. Setup

In [ ]:
!apt-get -qq install -y fluidsynth > /dev/null
!pip install -q miditoolkit note_seq pyfluidsynth soundfile datasets "transformers[torch]>=4.46,<5" accelerate wandb

## 2. Settings

Put the MIDI files you want to train on in a Google Drive folder and point `MIDI_DIR` at it. Subfolders are searched too, so an extracted [Lakh MIDI Dataset](https://colinraffel.com/projects/lmd/) folder works as is.

In [ ]:
from pathlib import Path

MIDI_DIR = Path("/content/drive/MyDrive/lmd_midis")   # folder of .mid files to train on
WORK_DIR = Path("/content/drive/MyDrive/music_gpt2")  # token text, tokenizer, checkpoints and generated audio

MAX_NB_BAR = 8         # length of each training piece, in bars
MIN_NB_NOTES = 10      # drop pieces with fewer notes than this
STEPS_PER_BEAT = 4     # timing grid: sixteenth notes in 4/4
STEPS_PER_BAR = STEPS_PER_BEAT * 4
CONTEXT_LENGTH = 512   # tokens per training example

HF_USERNAME = None     # set to your Hugging Face username to publish the dataset and tokenizer

In [ ]:
from google.colab import drive

drive.mount("/content/drive")
WORK_DIR.mkdir(parents=True, exist_ok=True)

## 3. Cut MIDI files into 8-bar pieces

Each file is split every `MAX_NB_BAR` bars (assuming 4/4). Notes are assigned to the piece they start in, and their times are shifted so every piece starts at zero. Files shorter than two pieces, and pieces with fewer than `MIN_NB_NOTES` notes, are skipped.

In [ ]:
from copy import deepcopy
from math import ceil

from miditoolkit import MidiFile


def split_into_chunks(midi):
    """Cut a MidiFile into MAX_NB_BAR-bar pieces, with note times relative to each piece."""
    ticks_per_cut = MAX_NB_BAR * midi.ticks_per_beat * 4
    nb_cuts = ceil(midi.max_tick / ticks_per_cut)

    template = deepcopy(midi)
    for track in template.instruments:
        track.notes = []
    chunks = [deepcopy(template) for _ in range(nb_cuts)]

    for i, track in enumerate(midi.instruments):
        for note in sorted(track.notes, key=lambda n: n.start):
            cut_id = note.start // ticks_per_cut
            if cut_id >= nb_cuts:
                continue
            note = deepcopy(note)
            note.start -= cut_id * ticks_per_cut
            note.end -= cut_id * ticks_per_cut
            chunks[cut_id].instruments[i].notes.append(note)
    return chunks

## 4. Write each piece as text

Pieces are encoded in the MMM (Multi-Track Music Machine) format: tracks one after another, each made of bars, each bar a list of note events.

```
PIECE_START
  TRACK_START INST=0 DENSITY=2
    BAR_START NOTE_ON=60 TIME_DELTA=4 NOTE_OFF=60 TIME_DELTA=2 NOTE_ON=64 TIME_DELTA=2 NOTE_OFF=64 BAR_END
    BAR_START ... BAR_END
  TRACK_END
  TRACK_START INST=33 ...
PIECE_END
```

* `INST` is the General MIDI program number.
* `DENSITY` is roughly the track's notes per beat.
* `TIME_DELTA` moves time forward in sixteenth notes. Times are snapped to that grid so the vocabulary stays small regardless of each file's resolution.
* Each track plays one note at a time. A note that starts while the previous one is still sounding is moved to just after it.

In [ ]:
def midi_to_tokens(midi):
    """Encode a MidiFile piece as an MMM token string on a sixteenth-note grid."""
    step = midi.ticks_per_beat / STEPS_PER_BEAT
    tokens = ["PIECE_START"]

    for track in midi.instruments:
        if not track.notes:
            continue
        density = len(track.notes) // (MAX_NB_BAR * 4)
        tokens += ["TRACK_START", f"INST={track.program}", f"DENSITY={density}"]
        notes = sorted(track.notes, key=lambda n: n.start)

        for bar in range(MAX_NB_BAR):
            bar_start, bar_end = bar * STEPS_PER_BAR, (bar + 1) * STEPS_PER_BAR
            tokens.append("BAR_START")
            t = bar_start
            for note in notes:
                start = round(note.start / step)
                if not bar_start <= start < bar_end:
                    continue
                start = max(start, t)  # one note at a time per track
                if start >= bar_end:
                    continue
                end = min(max(round(note.end / step), start + 1), bar_end)
                if start > t:
                    tokens.append(f"TIME_DELTA={start - t}")
                tokens += [f"NOTE_ON={note.pitch}", f"TIME_DELTA={end - start}", f"NOTE_OFF={note.pitch}"]
                t = end
            tokens.append("BAR_END")

        tokens.append("TRACK_END")

    tokens.append("PIECE_END")
    return " ".join(tokens)

In [ ]:
from tqdm.auto import tqdm

pieces_path = WORK_DIR / "pieces.txt"
midi_paths = sorted(p for p in MIDI_DIR.rglob("*") if p.suffix.lower() in (".mid", ".midi"))
print(f"{len(midi_paths)} MIDI files found in {MIDI_DIR}")

nb_pieces = 0
with open(pieces_path, "w") as f:
    for path in tqdm(midi_paths, desc="Cutting and tokenizing"):
        try:
            midi = MidiFile(str(path))
        except Exception as e:
            print(f"Skipping {path.name}: {e}")
            continue
        chunks = split_into_chunks(midi)
        if len(chunks) < 2:
            continue  # shorter than MAX_NB_BAR bars
        for chunk in chunks:
            if sum(len(track.notes) for track in chunk.instruments) >= MIN_NB_NOTES:
                f.write(midi_to_tokens(chunk) + "\n")
                nb_pieces += 1

print(f"{nb_pieces} pieces saved to {pieces_path}")
with open(pieces_path) as f:
    first_piece = f.readline().strip()
print(first_piece[:300])

## 5. Build the dataset

In [ ]:
from datasets import load_dataset

dataset = load_dataset("text", data_files=str(pieces_path), split="train")
dataset = dataset.train_test_split(test_size=0.1, shuffle=True, seed=42)
dataset

## 6. Train a word-level tokenizer

GPT-2's own tokenizer was built for English, so it breaks music tokens into meaningless fragments:

In [ ]:
from transformers import AutoTokenizer

sample = first_piece[:120]
print(AutoTokenizer.from_pretrained("gpt2").tokenize(sample))

Instead, every distinct MMM token becomes one word in the vocabulary. The pre-tokenizer has to be `WhitespaceSplit`: the plain `Whitespace` one also splits on `=`, turning `NOTE_ON=60` into `NOTE_ON`, `=`, `60`, and generated text could no longer be turned back into notes.

In [ ]:
from tokenizers import Tokenizer
from tokenizers.models import WordLevel
from tokenizers.pre_tokenizers import WhitespaceSplit
from tokenizers.trainers import WordLevelTrainer
from transformers import PreTrainedTokenizerFast


def training_corpus():
    for i in range(0, len(dataset["train"]), 1000):
        yield dataset["train"][i : i + 1000]["text"]


word_tokenizer = Tokenizer(WordLevel(unk_token="[UNK]"))
word_tokenizer.pre_tokenizer = WhitespaceSplit()
word_tokenizer.train_from_iterator(training_corpus(), WordLevelTrainer(special_tokens=["[UNK]", "[PAD]"]))

tokenizer = PreTrainedTokenizerFast(tokenizer_object=word_tokenizer, unk_token="[UNK]", pad_token="[PAD]")
tokenizer.save_pretrained(str(WORK_DIR / "tokenizer"))
print(f"Vocabulary: {len(tokenizer)} tokens")
print(tokenizer.tokenize(sample))

Optional: publish the dataset and tokenizer to the Hugging Face Hub (set `HF_USERNAME` in the settings cell).

In [ ]:
if HF_USERNAME:
    from huggingface_hub import notebook_login

    notebook_login()
    dataset.push_to_hub(f"{HF_USERNAME}/mmm_lmd_8bars")
    tokenizer.push_to_hub(f"{HF_USERNAME}/mmm_lmd_8bars_tokenizer")

## 7. Tokenize the dataset

Pieces longer than `CONTEXT_LENGTH` tokens are truncated, which mostly cuts the last tracks of busy multi-track pieces.

In [ ]:
def tokenize(batch):
    return {"input_ids": tokenizer(batch["text"], truncation=True, max_length=CONTEXT_LENGTH)["input_ids"]}


tokenized = dataset.map(tokenize, batched=True, remove_columns=["text"])
tokenized

## 8. Model

A GPT-2 with 6 layers, 8 attention heads and 512-dimensional embeddings, trained from random weights. The collator pads each batch and sets up next-token prediction (`mlm=False`).

In [ ]:
from transformers import DataCollatorForLanguageModeling, GPT2Config, GPT2LMHeadModel

config = GPT2Config(
    vocab_size=len(tokenizer),
    n_positions=CONTEXT_LENGTH,
    n_layer=6,
    n_head=8,
    n_embd=512,
    pad_token_id=tokenizer.pad_token_id,
    bos_token_id=tokenizer.convert_tokens_to_ids("PIECE_START"),
    eos_token_id=tokenizer.convert_tokens_to_ids("PIECE_END"),
)
model = GPT2LMHeadModel(config)
print(f"{model.num_parameters() / 1e6:.1f}M parameters")

data_collator = DataCollatorForLanguageModeling(tokenizer, mlm=False)

## 9. From tokens back to music

The decoder reverses the encoding: it restarts the clock at each track, jumps to the bar line at each `BAR_START`, and closes any note the model forgot to end. Playback assumes 120 BPM.

Decoding the first training piece and listening to it is a quick check that the encoding is correct before any training happens.

In [ ]:
import note_seq
import soundfile as sf
from IPython.display import Audio, display

SAMPLE_RATE = 44100
SECONDS_PER_STEP = 60 / 120 / STEPS_PER_BEAT


def tokens_to_note_sequence(text):
    """Decode an MMM token string into a NoteSequence that can be played or saved as MIDI."""
    ns = note_seq.NoteSequence(ticks_per_quarter=480)
    ns.tempos.add(qpm=120)
    track, program, bar, t = -1, 0, -1, 0
    open_notes = {}

    def close_open_notes():
        for note in open_notes.values():
            note.end_time = note.start_time + SECONDS_PER_STEP
        open_notes.clear()

    for token in text.split():
        name, _, value = token.partition("=")
        if name == "TRACK_START":
            close_open_notes()
            track, program, bar, t = track + 1, 0, -1, 0
        elif name == "INST":
            program = int(value)
        elif name == "BAR_START":
            bar += 1
            t = bar * STEPS_PER_BAR
        elif name == "TIME_DELTA":
            t += int(value)
        elif name == "NOTE_ON":
            close_open_notes()  # tracks are one note at a time
            open_notes[int(value)] = ns.notes.add(
                pitch=int(value), velocity=80, program=program, instrument=max(track, 0),
                start_time=t * SECONDS_PER_STEP,
            )
        elif name == "NOTE_OFF":
            note = open_notes.pop(int(value), None)
            if note is not None:
                note.end_time = max(t * SECONDS_PER_STEP, note.start_time + SECONDS_PER_STEP)
        elif name == "PIECE_END":
            break

    close_open_notes()
    ns.total_time = max((note.end_time for note in ns.notes), default=0)
    return ns


def save_and_play(ns, path):
    """Write a NoteSequence as .wav and .mid, and play it in the notebook."""
    audio = note_seq.fluidsynth(ns, sample_rate=SAMPLE_RATE)
    sf.write(str(path), audio, SAMPLE_RATE)
    note_seq.sequence_proto_to_midi_file(ns, str(path.with_suffix(".mid")))
    display(Audio(str(path)))


GENERATED_DIR = WORK_DIR / "generated"
GENERATED_DIR.mkdir(exist_ok=True)
save_and_play(tokens_to_note_sequence(first_piece), GENERATED_DIR / "training_example.wav")

## 10. Train

`MusicTrainer` composes a piece after every evaluation and saves it to `generated/` as `.wav` and `.mid`, so you can hear the model go from noise to music.

To log to Weights & Biases, add your API key as a secret named `WANDB_API_KEY` in Colab's Secrets panel (the key icon on the left). Without it, training runs the same and just doesn't log there.

In [ ]:
import os

import torch
from transformers import Trainer, TrainingArguments

try:
    from google.colab import userdata

    os.environ["WANDB_API_KEY"] = userdata.get("WANDB_API_KEY")
except Exception:
    pass

PROMPT = "PIECE_START TRACK_START"


def compose(model, max_new_tokens=CONTEXT_LENGTH - 2, temperature=0.75):
    """Sample a new piece from the model and return it as token text."""
    model.eval()
    input_ids = tokenizer(PROMPT, return_tensors="pt").input_ids.to(model.device)
    with torch.no_grad():
        output = model.generate(
            input_ids,
            max_new_tokens=max_new_tokens,
            do_sample=True,
            temperature=temperature,
            eos_token_id=config.eos_token_id,
            pad_token_id=tokenizer.pad_token_id,
        )
    return tokenizer.decode(output[0], skip_special_tokens=True)


class MusicTrainer(Trainer):
    """Trainer that composes and saves a piece after every evaluation."""

    def evaluation_loop(self, *args, **kwargs):
        output = super().evaluation_loop(*args, **kwargs)
        self.compose_sample(f"step_{self.state.global_step}")
        return output

    def compose_sample(self, name):
        ns = tokens_to_note_sequence(compose(self.model))
        if not ns.notes:
            print(f"{name}: no notes generated yet")
            return
        print(f"{name}: {len(ns.notes)} notes, {ns.total_time:.1f} s")
        save_and_play(ns, GENERATED_DIR / f"{name}.wav")


training_args = TrainingArguments(
    output_dir=str(WORK_DIR / "checkpoints"),
    num_train_epochs=3,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=64,
    warmup_steps=500,
    weight_decay=0.01,
    logging_steps=100,
    eval_strategy="steps",
    eval_steps=500,
    save_steps=1000,
    save_total_limit=2,
    fp16=torch.cuda.is_available(),
    report_to="wandb" if os.environ.get("WANDB_API_KEY") else "none",
)

trainer = MusicTrainer(
    model=model,
    args=training_args,
    data_collator=data_collator,
    train_dataset=tokenized["train"],
    eval_dataset=tokenized["test"],
    processing_class=tokenizer,
)
trainer.train()
trainer.save_model(str(WORK_DIR / "model"))

## 11. Compose

In [ ]:
for i in range(3):
    trainer.compose_sample(f"final_{i}")